# Downloading MODIS MOD44B Percent_Tree_Cover (Year 2000)

Validation layer for the Hansen forest-loss measure.

Source: [MODIS/061/MOD44B](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD44B) — Vegetation Continuous Fields, yearly, native 250m.

Exports are sent to Google Drive via Earth Engine. Monitor progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import ee
import geemap

ee.Authenticate()  # run once if not already authenticated
ee.Initialize()

In [3]:
# Global extent (same bbox used for the Hansen export)
world_bbox = ee.Geometry.BBox(-180, -85, 180, 85)

# MOD44B is a yearly ImageCollection. The year-2000 image has start date 2000-03-05.
mod44b_2000 = (
    ee.ImageCollection("MODIS/061/MOD44B")
    .filterDate("2000-01-01", "2000-12-31")
    .select("Percent_Tree_Cover")
    .first()
)

print(mod44b_2000.getInfo()["id"])  # sanity-check the image id

MODIS/061/MOD44B/2000_03_05


In [4]:
# Quick preview
vis_params = {"min": 0, "max": 100, "palette": ["ffffff", "006400"]}

Map = geemap.Map(center=[0, 0], zoom=2)
Map.addLayer(mod44b_2000.clip(world_bbox), vis_params, "MOD44B Percent_Tree_Cover 2000")
Map.addLayer(world_bbox, {}, "Region")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
# Export to Google Drive at native 250m
task = ee.batch.Export.image.toDrive(
    image=mod44b_2000,
    description="MOD44B_PercentTreeCover_2000_250m",
    folder="GEE_exports",
    fileNamePrefix="MOD44B_PercentTreeCover_2000_250m",
    region=world_bbox,
    scale=250,
    crs="EPSG:4326",
    maxPixels=1e13,
)

task.start()
print("Export task started:", task.id)

Export task started: 7MNXW2HFSZ2P5FASAE3NWLNO


### NOTE: track the export at the [GEE Task Manager](https://code.earthengine.google.com/tasks)

Once the tiles finish, drop them into `maps/raw/MOD44B/` in the Dropbox project folder to mirror the Hansen layout.